In [37]:
# 데이터 준비 (파일에서 배치를 적용한 읽기 도구 만들기)

from tensorflow import keras as tf_keras

train_dataset = tf_keras.utils.text_dataset_from_directory(
    'data-files/aclimdb/train', batch_size=32
)
validation_dataset = tf_keras.utils.text_dataset_from_directory(
    'data-files/aclimdb/val', batch_size=32
)
test_dataset = tf_keras.utils.text_dataset_from_directory(
    'data-files/aclimdb/test', batch_size=32
)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


In [2]:
# BoW 모델 기반 텍스트 데이터 인코딩 도구 학습

max_length = 600 # 한 문장의 길이 (토큰 갯수)
max_tokens = 20000 # 단어 사전의 크기 (토큰 갯수)
text_vectorization = tf_keras.layers.TextVectorization(
    max_tokens=max_tokens, # 단어 사전에 포함될 단어 갯수 (빈도수 높은 순)
    output_mode='int', # 각 단어의 단어 사전에 지정된 번호 인코딩
    output_sequence_length=max_length # 한 문장을 구성하는 단어 갯수
)

only_text_dataset = train_dataset.map(lambda x, y: x)
text_vectorization.adapt( only_text_dataset ) # 학습을 통해 단어 사전 구성

In [38]:
# 데이터 셋의 각 데이터에 대해 인코딩 처리

encoded_train_dataset = train_dataset.map( lambda x, y: (text_vectorization(x), y), num_parallel_calls=4 )
encoded_validation_dataset = validation_dataset.map( lambda x, y: (text_vectorization(x), y), num_parallel_calls=4 )
encoded_test_dataset = test_dataset.map( lambda x, y: (text_vectorization(x), y), num_parallel_calls=4 )

In [39]:
# 인코딩 결과 확인

for features, targets in encoded_train_dataset:
    print( features[0].shape, features[0][:10], features[0][-10:] )
    print( features[1].shape, features[1][:10], features[1][-10:] )
    break

(600,) tf.Tensor([ 10  66  59 315  49  11  20  14  43  41], shape=(10,), dtype=int64) tf.Tensor([0 0 0 0 0 0 0 0 0 0], shape=(10,), dtype=int64)
(600,) tf.Tensor([1326  409 1326   11    7    4   18   43  105  324], shape=(10,), dtype=int64) tf.Tensor([0 0 0 0 0 0 0 0 0 0], shape=(10,), dtype=int64)


In [44]:
import tensorflow as tf

inputs = tf_keras.layers.Input(shape=(None,), dtype='int64')
embedding = tf_keras.layers.Lambda(
    lambda x: tf.one_hot(x, depth=max_tokens),
    output_shape=lambda input_shape: (None, None, max_tokens)
)(inputs)
x = tf_keras.layers.LSTM(32)(embedding)
outputs = tf_keras.layers.Dense(1, activation='sigmoid')(x)

# embedding_model = tf_keras.Model(inputs, embedding)
# embedding_model(features)

model = tf_keras.Model(inputs, outputs)
model.compile(loss='binary_crossentropy',
              optimizer=tf_keras.optimizers.Adam(learning_rate=0.0001),
              metrics=['accuracy'])
# model.summary()

# 모델 저장 콜백 만들기
callbacks = [
    tf_keras.callbacks.ModelCheckpoint('models/imdb-lstm-model-1.keras', save_best_only=True)
]

# 모델 훈련
history = model.fit(encoded_train_dataset, epochs=10, 
                    validation_data=encoded_validation_dataset, 
                    callbacks=callbacks)

Epoch 1/10
  2/625 ━━━━━━━━━━━━━━━━━━━━ 42:27 4s/step - accuracy: 0.4922 - loss: 0.6932  

KeyboardInterrupt: 